# Earnings & Thesis Reviewer

Was hat sich seit dem Jahresbericht veraendert, und haelt die These noch? Grundlage: der letzte Annual Report des Agenten (Key Takeaways = Thesen), eine Scorecard (letztes Quartal vs. Vorjahresquartal vs. Geschaeftsjahr), die Pressemitteilung zu den Zahlen (8-K, EDGAR, kostenlos) und das 10-Q (2 sec-api-Calls pro Quartal, danach Cache).

**Voraussetzung:** Fuer die Firma existiert ein Annual Report des Agenten (neuer Lauf nach dem 20.11., damit die Argumente gespeichert sind; fuer aeltere Reports reicht eine `*_build_args.json` im Report-Ordner).

In [ ]:
# 1) Pfade und Imports
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import autogen
from finrobot.utils import register_keys_from_json
from lib.earnings_reviewer import run_review, reviewer_message
from lib.quarterly_data import scorecard_text


In [ ]:
# 2) Modell und Schluessel
MODEL = "anthropic/claude-haiku-5.5"

base = json.load(open(FR / "OAI_CONFIG_LIST", encoding="utf-8"))
entry = next((e for e in base if e["model"] == MODEL), None) or dict(base[0], model=MODEL)
llm_config = {"config_list": [entry], "timeout": 300, "temperature": 0.3,
              "parallel_tool_calls": False, "max_tokens": 12000}
if MODEL.startswith("google/"):
    llm_config["extra_body"] = {"reasoning": {"effort": "low"}}
register_keys_from_json(str(FR / "config_api_keys"))


In [ ]:
# 3) Firma und Geschaeftsjahr des Annual Reports (Ausgangsbasis)
COMPANY = "RKLB"
FYEAR = "2025"
save_dir = f"../reports/{COMPANY}_earnings_{MODEL.split('/')[-1]}"
print(scorecard_text(COMPANY, FYEAR))   # Vorschau der Scorecard (ohne LLM)


In [ ]:
# 4) Agent starten
run_review(llm_config, COMPANY, FYEAR, save_dir)


In [ ]:
# 5) PDF anzeigen
import io, glob
import pymupdf as fitz
from PIL import Image
pdf = glob.glob(f"{save_dir}/*_earnings_update.pdf")
if pdf:
    for page in fitz.open(pdf[0]):
        display(Image.open(io.BytesIO(page.get_pixmap(dpi=100).tobytes("png"))))
else:
    print("Kein PDF erzeugt - siehe Ausgabe von Zelle 4.")
